<style>
    /* we can use injections to style the resulting pdf */

    /* wrapping */
    pre, .jp-CodeCell .jp-Editor {
        display: block!important;
    }
    .jp-Cell-inputArea pre {
           page-break-inside: avoid !important;
    }
    .cm-editor.cm-s-jupyter .highlight pre {
        white-space: pre-wrap !important;
    }

    /* margins */
    div#notebook-container,
    div.container,
    div#notebook,
    .jp-Notebook {
        max-width: none !important;
        width: 102.6% !important;
        margin-left: -2.6% !important;
        padding: 0 !important;
    }
    .jp-MarkdownCell {
        margin-left: -6px;
        margin-bottom: 8px;
        margin-top: 12px;
    }
    .jp-OutputArea .jp-RenderedText {
        padding-left: calc(1ch + 16px);
        padding-top: 4px;
        padding-bottom: 8px;
    }
</style>

In [1]:
import pandas as pd
import numpy as np
import re
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
import unicodedata

C:\Users\windo\PycharmProjects\PythonProject1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. PREPARATIONS

In [3]:
clean_comments = pd.read_parquet('clean_comments.parquet')

In [4]:
clean_submissions = pd.read_parquet('clean_submissions.parquet')

In [5]:
t1_comments_final = pd.read_parquet('t1_comments_final.parquet')
t2_comments_final = pd.read_parquet('t2_comments_final.parquet')
t3_comments_final = pd.read_parquet('t3_comments_final.parquet')

In [6]:
t1_submissions_final = pd.read_parquet('t1_submissions_final.parquet')
t2_submissions_final = pd.read_parquet('t2_submissions_final.parquet')
t3_submissions_final = pd.read_parquet('t3_submissions_final.parquet')

The first step in this stage of the research involves conducting a general BERTopic analysis of the entire (filtered) dataset. We decided to combine comments and submissions within this step (it can be assumed that from a semantic perspective, a “title/selftext” describing a problem and a “comment” describing a solution exist in the same vector space). Thus, the resulting dataset includes:
* Filtered datasets for tier 2/tier 3 + tier 1 r/LocaLLlama (t1_comments_final format) for comments and submissions
* The clean_comments and clean_submissions datasets, from which we extract tier 1 communities (except for r/LocaLLlama).

At the same time, the final dataset obtained in this way also requires additional filtering. Its main volume is occupied by the CharacterAI and JanitorAI_Official communities, with 624,789 and 155,426 records, respectively.

From the previous stage of analysis, we discovered that these subreddits are structural outliers with a significantly shorter word count distribution. Considering the reaction-sharing nature, we decided to apply a minimum length threshold to such high-volume roleplay subreddits, which allowed us to focus on analyzing a more substantive discourse without losing the intimate/emotional signal. We do not perform similar filtering for the remaining tier 1 subreddits, given their word count distribution, popularity (unlike JanitorAI_Official and CharacterAI, which are among the most popular subreddits in the dataset), and the general nature of the discussions.

In [7]:
# the filtered datasets (tier 2 filtered, tier 3 filtered, tier 1 LocalLlama filtered)

In [8]:
df_list = [
    t1_comments_final, t1_submissions_final,
    t2_comments_final, t2_submissions_final,
    t3_comments_final, t3_submissions_final
]

In [9]:
CHAT_SUBS = ['CharacterAI', 'JanitorAI_Official']
UNFILTERED_TARGETS = ['SillyTavernAI', 'JanitorAI_Official', 'CharacterAI', 'MyBoyfriendIsAI']

MIN_WORD_COUNT = 25

In [10]:
# standardizing columns before merging, we keep a metadata column source_type
for df in df_list:
    if 'body' not in df.columns and 'title' in df.columns:
        df['text'] = df['title'].fillna('') + " " + df['selftext'].fillna('')
        df['source_type'] = 'submission'
    else:
        df['text'] = df['body']
        df['source_type'] = 'comment'

In [11]:
df_filtered = pd.concat(df_list, ignore_index=True)
df_filtered['origin'] = 'filtered_set'

In [12]:
# the raw datasets

In [13]:
# comments
mask_unfiltered_c = clean_comments['subreddit'].isin(UNFILTERED_TARGETS)
df_unfiltered_c = clean_comments[mask_unfiltered_c].copy()
df_unfiltered_c['text'] = df_unfiltered_c['body']
df_unfiltered_c['source_type'] = 'comment'

In [14]:
# length filter to cai and janitor
mask_short = (df_unfiltered_c['subreddit'].isin(CHAT_SUBS)) & (df_unfiltered_c['word_count'] < MIN_WORD_COUNT)
print(f"Dropping {mask_short.sum()} short comments")
df_unfiltered_c = df_unfiltered_c[~mask_short]

Dropping 555272 short comments


In [15]:
# submissions
mask_unfiltered_s = clean_submissions['subreddit'].isin(UNFILTERED_TARGETS)
df_unfiltered_s = clean_submissions[mask_unfiltered_s].copy()
df_unfiltered_s['text'] = df_unfiltered_s['title'].fillna('') + " " + df_unfiltered_s['selftext'].fillna('')
df_unfiltered_s['source_type'] = 'submission'

In [16]:
df_unfiltered = pd.concat([df_unfiltered_c, df_unfiltered_s], ignore_index=True)
df_unfiltered['origin'] = 'full_set'

In [17]:
# final merge

In [18]:
final_df = pd.concat([df_filtered, df_unfiltered], ignore_index=True)

In [19]:
final_df['subreddit'].value_counts()

subreddit
CharacterAI              210622
JanitorAI_Official        81450
SillyTavernAI             37891
LocalLLaMA                12483
ChatGPT                    8108
singularity                3927
MyBoyfriendIsAI            3862
lonely                     2003
ArtificialInteligence      1189
FanFiction                  223
Name: count, dtype: int64

In [21]:
# aggressive normalization can strip the tone that we are trying to capture,
# we do some light cleaning to filter out the general noice

def light_clean(text):
    if not isinstance(text, str): return ""
    # URLs
    text = re.sub(r'https?://\S+', '', text)
    # user mentions and subreddits
    text = re.sub(r'u/[A-Za-z0-9_-]+', '', text)
    text = re.sub(r'r/[A-Za-z0-9_-]+', '', text)
    # newlines
    text = text.replace('\n', ' ')
    # spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # NFKC
    text = unicodedata.normalize('NFKC', text)

    return text

In [22]:
final_df['text_clean'] = final_df['text'].apply(light_clean)

In [23]:
print(f"Final Dataset Size: {len(final_df)}")
print(final_df['subreddit'].value_counts())

Final Dataset Size: 361758
subreddit
CharacterAI              210622
JanitorAI_Official        81450
SillyTavernAI             37891
LocalLLaMA                12483
ChatGPT                    8108
singularity                3927
MyBoyfriendIsAI            3862
lonely                     2003
ArtificialInteligence      1189
FanFiction                  223
Name: count, dtype: int64


In [76]:
# given the special nature of r/MyBoyfriendIsAI, in certain places we may apply
# similar operations to it as we do for the Tier 1 subreddits. at the same time,
# conceptually, we still consider it a Tier 3 subreddit

tier_map = {
    'CharacterAI': 'Tier 1',
    'JanitorAI_Official': 'Tier 1',
    'SillyTavernAI': 'Tier 1',
    'LocalLLaMA': 'Tier 1',

    'ChatGPT': 'Tier 2',
    'ArtificialInteligence': 'Tier 2',
    'singularity': 'Tier 2',

    'FanFiction': 'Tier 3',
    'lonely': 'Tier 3',
    'MyBoyfriendIsAI': 'Tier 3',
}

final_df['tier'] = final_df['subreddit'].map(tier_map)

In [44]:
print(final_df['tier'].value_counts())

tier
Tier 1    342446
Tier 2     13224
Tier 3      6088
Name: count, dtype: int64


We can see a notable imbalance between the sizes of tiers/subreddits. At the same time, since at this stage we are conducting more of a landscape analysis and a general overview, this is acceptable. Given that BERTopic is density-based, we can expect that tier 3 documents will likely get absorbed into larger tier 1 documents

In [ ]:
# 2. EMBEDDINGS

In [2]:
# considering the size of the dataset, as well as hardware and time constraints,
# it was decided to use all-MiniLM-L6-v2 for speed/performance trade-off
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

In [25]:
embeddings = model.encode(
    final_df['text_clean'].tolist(),
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

Batches: 100%|██████████| 5653/5653 [28:35<00:00,  3.30it/s] 


In [26]:
np.save("embeddings_landscape.npy", embeddings)

In [27]:
final_df.to_parquet("dataset_landscape.parquet")

In [ ]:
# 3. MODEL TRAINING

Initial analysis with standard parameters revealed the necessity of tuning those. Although the model detected a certain amount of niche subculture dynamics, the results also had some structural failures.

Despite the fact that more than 200,000 documents were classified as Topic -1, i.e., outliers, their representative documents had high-quality and technically rich thematic posts on topics consistent with those already identified by the model. Topic 0 also did not have a specific theme and combined more than 45,000 documents on the topic of “communication” with models in a broad sense.

At the same time, the model successfully identified, for example, a specific hallucination issue in CharacterAI LLMs (“I felt a pang of pang”), separated popular role-playing topics, and highlighted the release of DeepSeek v3/r1 as a separate topic.

As a result of iterative analysis, we reached the following parameters:
* It was decided force the model to look at a wider neighborhood (n_neighbors=30). Given the sparse (short, slang-heavy) nature of Reddit comments, the increased n_neighbors could allow the model to consider messages in a broader context (e.g., ignoring the use of different slang and subcultural expressions).
* It was decided to keep a granular cluster size (60) but lower the min_samples to 5. In general, and especially from comments, we can expect a sharp change in the topics and moods within the message - and a low min_samples value will allow us to separate similar documents from Topic -1 and assign them to a specific category.
* It was decided to set nr_topics=None, to stop BERTopic from automatically merging similar topics, and then perform hierarchical reduction based on reduce_topics.



In [2]:
df = pd.read_parquet("dataset_landscape.parquet")
embeddings = np.load("embeddings_landscape.npy")

In [3]:
final_df = df.copy()

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

In [82]:
docs = df['text_clean'].tolist()
print(f"Loaded {len(docs)} documents")

Loaded 361758 documents


In [6]:
# contextual stopwords for labels
custom_stopwords = list(CountVectorizer(stop_words="english").get_stop_words())
domain_stopwords = [
    'ai', 'bot', 'chatbot', 'chatgpt', 'gpt', 'llm', 'model',
    'character', 'characterai', 'cai', 'janitor', 'janitorai',
    'just', 'like', 'really', 'people', 'use', 'make', 'know',
    'time', 'think', 'good', 'want', 'did', 'does',

    # artifacts
    'im', 'dont', 've', 'don', 'll', 're', 'm', 's', 't', 'isn', 'didn',

    # generic noise
    'chat', 'message', 'messages', 'talk', 'talking', 'say', 'said'
]
custom_stopwords.extend(domain_stopwords)

In [7]:
# dimensionality reduction
umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=1
)

# clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=60,
    min_samples = 5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# vectorizer
vectorizer_model = CountVectorizer(stop_words=custom_stopwords, ngram_range=(1, 2))

# for better labels, we extract keywords based on semantic similarity to the cluster
representation_model = {
    "KeyBERT": KeyBERTInspired(),
    "MMR": MaximalMarginalRelevance(diversity=0.3)
}

# BERTopic
topic_model = BERTopic(
    embedding_model=model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    nr_topics=None,
    verbose=True
)

In [ ]:
topics, probs = topic_model.fit_transform(docs, embeddings)

The resulting dataset has 407 topics (topic -1 as outlier and 406 main topics). We manually verified the first 200 topics - below, we will highlight and combine those into logical groups and mark topics  that provide the most valuable insights:

Hardware, models, and releases topics:
* Topic 8 (Deepseek): discussion about the release and usage of Deepseek V3/R1 as a cost-effective and less restricted alternative to OpenAI
* Topic 28 (hardware/vram): technical constraints and advice on running local models (16GB vs 24GB vram) to avoid censorship
* Topic 38 (GPT4/Turbo): discussions on the degradation of GPT4 quality and the cost of API usage
* Topic 68 (NovelAI): discussions regarding NovelAI’s Kayra model versus chatbot alternatives.
* Topic 85 (KoboldAI): troubleshooting the KoboldCPP backend, especially for connecting local models to frontends like SillyTavern
* Topic 145 (Llama 3): the arrival of Meta’s Llama 3 (8B) as a major shift in local roleplay quality
* **Topic 149 (censored/uncensored):** debates on fine-tuning Llama 2/3 to remove refusals and safety guardrails
* Topic 194 (70B/7B): the trade-off between model intelligence (70B parameters) and speed/cost (7B parameters)

Technical advises and general discource topics:
* **Topic 0 (age/minors)**: discussions about the presence of minors in NSFW spaces and the demand for age verification. It is important to note that this topic mainly occurs in C.AI, where users link their dissatisfaction with censorship to the presence of minors in their community. Given the size of this topic, it is clear that the restrictions caused by this are one of the most critical issues for C.AI users - we can consider this one of the most popular reasons for migration and the choice of alternatives by C.AI users.
* Topic 5 (ooc commands): users developing out of character formatting syntax to steer bot behavior
* Topic 18, 65, 162, 190 (jailbreaking): DAN prompts and logic traps to bypass NSFW filters on ChatGPT and C.AI
* Topic 26 (character definitions): best practices for writing character cards (JSON formats, example dialogue) to ensure bot consistency
* Topic 36 (reverse proxies): methods for stealing or sharing API keys to access paid models for free
* Topic 61 (temperature): tuning the temperature to balance creativity and logical coherence in responses
* Topic 120 (context window): discussions on RoPE scaling and extending context length so bots don't forget the story

Intimacy, therapy, and unintended uses topics:
* **Topic 1 (therapy/loneliness)**: using AI as a therapist or friend to cope with extreme isolation and depression. One of the most important insights in our context is the popularity of this topic - we see that it occurs not only in subreddits such as lonely or MyBoyfriendIsAI, but also spreads widely to subreddits with more general discussions.
* Topic 2 (gay/transgender): Users discuss the popularity and quality of bots for WLW/MLM relationships and share tips for correctly role-playing transgender characters.
* Topic 16 and 55 (violence/torture): users testing boundaries regarding extreme violence or torture scenes
* Topic 47 (romance): dissatisfaction with the fact that even “slow burn” bots often immediately move on to romance or sexual scenes
* Topic 63 (yandere): high popularity of the yandere archetype - we can assume a community preference for high-intensity devotion?
* Topic 73 (taboo/incest): users testing boundaries regarding incest relationships and consensual non-consent
* Topic 93 and 96 (horny/kink): discussions on fetishes and complaints about bots initiating sex too aggressively
* Topic 98 (sexting services): missed spam? or discussion regarding sexting apps
* Topic 117 (crying/emotional): users crying or feeling genuine grief during roleplay scenarios

Community culture, complaints:
* Topic 3, 9, 101, 156 (site issues): massive clusters of complaints about C.AI servers crashing or being slow
* Topic 11, 139 (hallucinations): a specific model failure where C.AI bots overuse the phrase "a pang of...", which became a community meme
* Topic 89, 102, 108 (The Filter): constant negative discourse regarding the C.AI nsfw filter (the f!lt3r etc.)
* Topic 118, 178 (data hoarding): users trying to download/export their chat logs (fear of platform shutdowns or bans)

In [10]:
topic_model.get_topic_info()[['Topic', 'Count', 'Name', 'KeyBERT', 'Representative_Docs']].head(200)

,Topic,Count,Name,KeyBERT,Representative_Docs
0,-1,206971,-1_bots_characters_way_new,"[prompt, models, chats, conversation, text, me...",[I recommend that people experiment with other...
1,0,9674,0_minors_age_18_kids,"[minors, teenagers, teens, teen, adults, adult...","[I fully agree with you, most of the posts I s..."
2,1,6495,1_therapist_real_friends_therapy,"[loneliness, therapist, therapy, feel lonely, ...",[Psychopath that went insane thanks to lonelin...
3,2,4681,2_gender_male_female_gay,"[female bots, male bots, mlm bots, gender neut...",[I’ll go in circles with u again one last time...
4,3,4531,3_site_servers_devs_app,"[apps, new site, old site, app, old website, q...",[I'm addicted to Character.ai and can't seem t...
...,...,...,...,...,...
195,194,157,194_70b_7b_models_70b models,"[70b models, better 70b, models 70b, 70b recom...","[To me, the difference between the 120b and 70..."
196,195,157,195_browser_chrome_opera_gx,"[opera gx, browser opera, gx browser, using op...",[Why wont GX let me update? So we all know tha...
197,196,156,196_streaming_text streaming_immersive mode_tu...,"[streaming enabled, enable streaming, streamin...",[Is text streaming not working for anyone else...
198,197,156,197_rizz_rizzing_rizzed_rizz rizz,"[rizz bots, bots rizz, rizz challenge, rizz, t...",[I have to find these two bots so I can see th...


Manual verification confirms that the reduction did not compromise the quality of the resulting dataset. Below are the main new insights and changes compared to the previous iteration:


Intimacy, therapy, and unintended uses topics:
* Separate topics for therapist, real, sentient, and loneliness merged into topic 2 - users discuss the AI feeling real because it cures their loneliness/depression (AI feels more real than human partners/therapists etc.)
* In a similar context, the following topics emerged:
    * Topic 59 (grief/loss): using AI to cope with the death of real-life relatives
    * Topic 147 (cheating/NTR): ethical debates on whether roleplaying with an AI is cheating on a real partner
* In addition to topics 18 (violence/torture) and 29 (yandere/waifu), topic 144 (omegaverse) also emerged  - a very specific cluster regarding alpha/omega dynamics, knotting, and pheromones - we can see how fanfiction culture is influencing AI roleplaying
* At the same time, topic 65 (kissing/romance) discusses complaints about C.AI filters blocking innocent romantic gestures like kissing

Hardware, models, and releases topics:
* The topics of uncensored models and jailbreaking have been merged into more logical clusters - users are interested in uncensored finetunes of Llama 2/3 and Wizard/Vicuna models and complain about the “lobotomy” and degradation of popular models such as GPT-4 Turbo over time. Several fragmented topics about DAN, prompting, and bypassing are now merged into Topic 14 (and partly Topic 162).
* An interesting insider topic is Topic 88 (Termux/Android) - a dedicated sub-community running LLMs locally on Android phones via termux - a high technical effort for mobile roleplay.

Separately, we can highlight topics dedicated to bot configuration - in addition to Topic 4 on writing bot definitions (W++ format, JSON, example dialogue) and Topic 6 on OOC commands, the following emerged:
* Topic 9 (memory/pinning): managing the limited context window by pinning core memories so the bot doesn't forget
* Topic 63 (lorebooks): creating external “lorebooks” to inject world-building data into the chat context

Community culture and complaints topics are still a separate category - topic 1 (minors/age), topic 5 (site down), topic 12 (pang hallucination), etc.

In [ ]:
topic_model.reduce_topics(docs, nr_topics=150)

In [13]:
topic_model.get_topic_info()[['Topic', 'Count', 'Name', 'KeyBERT', 'Representative_Docs']].head(150)

,Topic,Count,Name,KeyBERT,Representative_Docs
0,-1,206971,-1_bots_characters_way_new,"[prompt, models, chats, conversation, free, te...",[I recommend that people experiment with other...
1,0,15340,0_bots_male_gender_female,"[male bots, female bots, bots, gender, charact...",[Character Rating Tags? So after finding out a...
2,1,13037,1_minors_age_18_kids,"[minors, teens, teenagers, adults, teen, adult...",[>Maybe if they didn’t advertise towards immat...
3,2,12997,2_human_real_feel_life,"[ais, self, loneliness, therapy, humanity, the...",[how to not be lonely? vent hi guys i hope eve...
4,3,6362,3_chats_group_group chats_rooms,"[group chats, old chats, chat2, legacy chats, ...",[Anyone know a good group chat bot? I don't me...
...,...,...,...,...,...
145,144,66,144_omega_alpha_omegaverse_alphas,"[male omegas, alphas omegas, alpha omega, omeg...","[In this world, people are born with secondary..."
146,145,66,145_monika_kotonoha_hologram_monika monika,"[monika, monika monika, monika actually, monik...",[When I'm playing this website there are multi...
147,146,62,146_marie_community manager_devs_marielovesmatcha,"[updates marie, marie promised, marie, marie p...",[It’s also Marie’s job to be the middleman bet...
148,147,61,147_cheating_cheated_partner_consider cheating,"[fictional cheating, cheating partner, conside...",[I think that if she feels like she is cheatin...


To reduce the number of outliers, we applied the embeddings method (assigning topic -1 to the nearest clusters), and a threshold of 0.55 was selected iteratively as one that provides an acceptable number of outliers while preserving the context of existing topics. Considering the changes:

Hardware, models, and releases topics:
* Topic 11 (Deepseek) remains distinct - as it did not get absorbed into general model talk, we can assume that Deepseek is a distinct brand in the user's mind
* Topic 35 (GPT4/Turbo) absorbed general complaints about "OpenAI" quality
* Topic 88 (termux): very clean cluster, the outliers assigned here are almost certainly correct (i.e. logs from Android phones)

Intimacy, therapy, and unintended uses topics:
* Topic 2 (real/human/feel/life) - combines therapy, loneliness, existential quetions and validation seeking
* Topic 59 (crying/grief) absorbed stories of users mourning real family members using AI
* Topic 144 (omegaverse) remains highly specific, which is interesting - the vocabulary (knot, heat, alpha) is so distinct that it resisted the merge

Bot configuration:
* Topic 4 (сharacter сreation): doubled in size, now includes general questions alongside more advanced JSON guides
* Topic 6 (OOC commands): merged with general roleplay formatting tips.
* Topic 14 (Jailbreaking): now includes general bypass discussions (not just DAN prompts)

Community culture, complaints:
* Topic 37 (fixed/issue) is a generic "it works now" cluster - although this is analytically weak, it is probably unavoidable

The most important insights here are:

* Despite the reduction, Deepseek (topic 11) and Llama 3 (topic 52) remained separate. Users do talk about specific releases - in our context, it shows some confirmation to the theory that both Llama 3 and DeepSeek are distinct cultural events.
* The fact that highly specific fanfiction tropes such as omegaverse (topic 144) remained after reduction (given the small size of fanfiction messages in the final dataset) confirms the fundamental influence of AI culture on AI interaction in a role-playing context. Although these tropes are difficult to implement and require complex prompting to enforce, there is still a distinct group of users who want to implement them.

In [14]:
current_topics = topic_model.topics_
new_topics = topic_model.reduce_outliers(
    docs,
    current_topics,
    strategy="embeddings",
    embeddings=embeddings,
    threshold=0.55
)

In [ ]:
topic_model.update_topics(
    docs,
    topics=new_topics,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model # we also update representations!!
)

In [16]:
topic_model.get_topic_info()[['Topic', 'Count', 'Name', 'KeyBERT', 'Representative_Docs']].head(150)

,Topic,Count,Name,KeyBERT,Representative_Docs
0,-1,114635,-1_models_way_got_love,"[models, tell, hope, based, characters, idea, ...",[I recommend that people experiment with other...
1,0,35305,0_bots_male_female_gender,"[bots, gender, chats, personality, female, wom...",[Character Rating Tags? So after finding out a...
2,1,13612,1_minors_age_18_kids,"[minors, teens, 18, 17, teenagers, age, undera...",[>Maybe if they didn’t advertise towards immat...
3,2,25114,2_real_human_feel_life,"[ais, bots, self, intelligence, conversation, ...",[how to not be lonely? vent hi guys i hope eve...
4,3,12553,3_chats_group_new_group chats,"[group chats, new chats, saved chats, old chat...",[Anyone know a good group chat bot? I don't me...
...,...,...,...,...,...
145,144,80,144_omega_alpha_omegaverse_alphas,"[male omegas, omegas alphas, alphas omegas, al...","[In this world, people are born with secondary..."
146,145,88,145_monika_kotonoha_ddlc_hologram,"[monika, monika monika, monika actually, monik...",[When I'm playing this website there are multi...
147,146,86,146_marie_community manager_devs_mod,"[marie, exactly marie, marie basically, marie ...",[It’s also Marie’s job to be the middleman bet...
148,147,156,147_cheating_partner_cheated_cheat,"[relationship cheating, cheating partner, pers...",[I think that if she feels like she is cheatin...


In [ ]:
topic_model.save("bertopic_master_150_final")

In [96]:
fig1 = topic_model.visualize_barchart(
    top_n_topics=None,
    topics=[2, 32, 5, 4],
    title="Distinct groups of AI Subculture: Intimacy, Hardware, Friction, and Engineering"
)
fig1.show()
fig1.write_html("bertopic_1.html")

In [97]:
classes = final_df['subreddit'].tolist()
topics_per_class = topic_model.topics_per_class(docs, classes=classes)

fig4 = topic_model.visualize_topics_per_class(
    topics_per_class,
    top_n_topics=None,
    topics=[2, 32, 63, 144],
    title="Stratification of Discourse by Community Tier"
)
fig4.show()
fig4.write_html("bertopic_4.html")

10it [01:34,  9.49s/it]


In [101]:
tech_topics = [4, 6, 9, 14, 30, 32, 63, 88, 128]

fig5 = topic_model.visualize_hierarchy(
    topics=tech_topics,
    title="The 'Folk Engineering' Skill Tree: How Technical Concepts Relate"
)
fig5.show()
fig5.write_html("bertopic_5.html")

In [102]:
fig6 = topic_model.visualize_barchart(
    topics=[2, 18, 59, 147],
    n_words=10,
    title="Emotional extremes: Therapy vs. Torture"
)
fig6.show()
fig6.write_html("bertopic_6.html")